# Grafos Dinámicos Temporales T-GNN — Dataset FDIC RIS

El problema que tratamos tiene dos dimensiones de dependencia que un modelo tabular no puede capturar. 

1. Una dependencia temporal, donde el estado de un banco en un trimestre `t` depende de su historia. Esto lo captura `TabPFN` parcialmente con la ventana deslizante, durante el entrenamiento y la evaluación y el `LSTM baseline` explícitamente.

2. Una dependencia relacional: el estado de un banco en un trimestre `t` depende del estado de sus vecinos en la red financiera. Un banco sano conectado a un _holding_ en dificultades tiene un perfil de riesgo distinto a un banco sano aislado, aunque sus ratios __CAMELS__ sean idénticos. Esto no lo captura ningún modelo tabular por construcción, ya que como vemos no tienen acceso a la estructura del grafo, que vienen dada en las bases de datos de `STRU` y `MERG`.

Una de nuestras hipotesis, y por la cual empleamos T-GCN en el TFM, es que el riesgo bancario tiene una componente sistémica no capturada por _features_ individuales, y que esa componente es observable en la estructura de relaciones entre entidades.


Primero empezemos definiendo que es un grafo estático.

> __Definición__:  Un grafo estatico, denotado como G = (V, E) donde V es el conjunto de nodos y $E\subseteq V × V$ el conjunto de aristas. es una estructura matemática que representa relaciones entre elementos, en la que el conjunto de vértices y el conjunto de aristas permanecen fijos durante el análisis.

El problema o la limitación de los grafos estaticos es que los vértices y las aristas no cambian con el tiempo y por tanto la estructura del grafo se considera invariable durante el estudio del problema. En nuestro caso de estudio necesitamos que tanto los vertices como las aristas varien según los datos temporales que tenemos, luego necesitamos un grafo dinamico o temporal, donde los nodos y las conexiones entre ellos pueden aparecer, desaparecer o modificarse con el tiempo.

> __Definición:__ Un grafo dinámico es un modelo de grafo cuya estructura evoluciona a lo largo del tiempo. En lugar de considerar un único conjunto fijo de nodos y aristas, el sistema se representa mediante una secuencia temporal de snapshots o grafos estáticos:
>
> $$
> G = \{G^1, G^2, \dots, G^T\}
> $$
>
> donde cada snapshot $G^t$, para $t \in \{1,\dots,T\}$, se define como
>
> $$
> G^t = (V^t, E^t, X^t, A^t)
> $$
>
> siendo:
>
> - $V^t$ el conjunto de vértices presentes en el instante $t$,
> - $E^t$ el conjunto de aristas presentes en el instante $t$,
> - $X^t$ la matriz de características nodales en el instante $t$, donde cada nodo puede poseer atributos o variables descriptivas,
> - $A^t$ la matriz de adyacencia asociada al grafo en el instante $t$, definida por
>
> $$
> A_{ij}^t =
> \begin{cases}
> 1 & \text{si existe una arista entre } i \text{ y } j \text{ en } t, \\
> 0 & \text{en caso contrario.}
> \end{cases}
> $$
>
> para el caso de grafos no ponderados.

En nuestro caso, cada snapshot temporal del grafo dinámico se define como $G^t = (V^t, E^t, X^t, A^t)$ donde:

- $V^t$ es el conjunto de bancos activos en el trimestre $t$. Un banco $i \in V^t$ si y solo si aparece en `panel_nodos` con `ACTIVE = 1` en dicho trimestre `t`. El conjunto de nodos varía temporalmente debido a quiebras, fusiones y adquisiciones bancarias, por lo que $|V^t|$ decrece monótonamente desde aproximadamente 6.100 bancos en $2016Q1$ hasta aproximadamente 4.400 bancos en $2025Q4$.

- $E^t$ es el conjunto de aristas activas en el instante $t$. Una arista $(i,j) \in E^t$ representa una relación financiera entre los bancos $i$ y $j$. En nuestro caso, las aristas se derivan de eventos `MERG`, de forma que existe una arista $(i,j)$ si el banco $i$ adquirió o se fusionó con el banco $j$ en el instante $t$ o en un instante anterior.

- $X^t \in \mathbb{R}^{|V^t| \times d}$ es la matriz de atributos nodales en el instante $t$, donde cada fila
$$
x_i^t \in \mathbb{R}^d
$$
representa el vector de características del banco $i$ en el trimestre $t$. En particular:

$$
x_i^t =
\left[
e_i^{t,\mathrm{desarrollo}}
\;\Vert\;
s_i^t
\right]
$$

donde:

- $e_i^{t,\mathrm{desarrollo}} \in \mathbb{R}^{192}$ es el embedding TabPFN del banco $i$ en $t$, extraído de `embeddings_desarrollo.parquet`,
- $s_i^t \in \mathbb{R}^k$ corresponde a los atributos estructurales y financieros obtenidos de `panel_nodos` y codificados numéricamente.

La dimensión total del espacio de características viene dada por

$$
d = 192 + k
$$

- $A^t \in \mathbb{R}^{|V^t| \times |V^t|}$ es la matriz de adyacencia asociada al snapshot temporal $G^t$. En el caso no ponderado:

$$
A_{ij}^t =
\begin{cases}
1 & \text{si } (i,j)\in E^t, \\
0 & \text{en caso contrario.}
\end{cases}
$$

La construcción exacta de $E^t$ y $A^t$ constituye una de las decisiones de diseño fundamentales del grafo.

La elección de que el snapshot sea discreto de forma trimestral es natural porque los datos son trimestrales por construcción y los eventos `MERG` tienen fecha efectiva `EFFDATE` que se puede mapear a trimestre sin pérdida de información relevante.

Las Graph Neural Networks (GNNs) constituyen el paradigma fundamental para el aprendizaje sobre datos estructurados en forma de grafo. Su formulación unificadora se basa en el esquema de Message Passing Neural Networks (MPNN) propuesto por Gilmer et al. (2017).

En este marco, la actualización del embedding del nodo $ i $ en la capa $ l $ se define como:
$$
m_i^{(l)} = \text{AGGREGATE}\left(\{h_j^{(l-1)} : j \in \mathcal{N}(i)\}\right)
$$
$$
h_i^{(l)} = \text{UPDATE}\left(h_i^{(l-1)}, m_i^{(l)}\right)
$$

donde $ \mathcal{N}(i) $ representa el vecindario del nodo $ i $, y $ h_i^{(l)} $ es la representación latente (embedding) del nodo en la capa $ l $. Una de las instanciaciones más utilizadas de este marco es el Graph Convolutional Network (GCN) de Kipf & Welling (2017), definido como:
$$
H^{(l+1)} = \sigma\left(\tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2} H^{(l)} W^{(l)}\right)
$$

donde:

- $ \tilde{A} = A + I $ es la matriz de adyacencia con auto-conexiones añadidas. La auto-conexión es necesaria porque sin ella el nodo no incluye su propio estado anterior en la agregación

- $ \tilde{D} $ es la matriz diagonal de grados de $ \tilde{A} $, con $ \tilde{D}_{ii} = \sum_j \tilde{A}_{ij} $.

- $ \tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2} $ es la normalización simétrica. Sin normalizar, nodos con muchos vecinos acumulan señal arbitrariamente grande. La normalización simétrica garantiza que la propagación preserve la escala de los features independientemente del grado del nodo

- $ H^{(l)} \in \mathbb{R}^{n \times d} $ representa los embeddings en la capa $ l $, con $ H^{(0)} = X $.

- $ W^{(l)} \in \mathbb{R}^{d \times d'} $ es la matriz de pesos entrenables de la capa $l$

- $ \sigma $ es una función de activación no lineal, típicamente ReLU.

Intuitivamente, cada capa GCN permite que cada nodo agregue información de sus vecinos directos, de modo que con $ L $ capas cada nodo incorpora información de su vecindario a distancia $ L $.



T-GCN — formalización de grafos dinámicos

El Temporal Graph Convolutional Network (T-GCN) propuesto por Zhao et al. (2019) integra GCNs con Gated Recurrent Units (GRU) para capturar simultáneamente dependencias espaciales (estructura del grafo) y temporales (evolución dinámica).

La GRU estándar (Cho et al., 2014) actualiza el estado oculto $h_t$ como:
$$
z_t = \sigma(W_z [h_{t-1}, x_t])
$$
$$
r_t = \sigma(W_r [h_{t-1}, x_t])
$$
$$
\tilde{h}_t = \tanh(W [r_t \odot h_{t-1}, x_t])
$$
$$
h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t
$$

donde $ z_t $ es la puerta de actualización que decide cuanto del estado anterior conservar, $ r_t $ la puerta de reset que decide cuánto del estado anterior olvidar al calcular el candidato $\tilde{h}_t$ y $ \odot $ el producto de Hadamard.

Se puede extender el modelo de GRU a grafos, lo cual da como resultado las T-GCN, donde estas sustituyen las transformaciones lineales de las GRU por operaciones de convolución GCN sobre grafos. Formalmente, T-GCN reemplaza la multiplicación matricial lineal $x_t$ en la GRU por una convolución GCN sobre el grafo en t:

$$
z_t = \sigma\left(W_z [\text{GCN}(h_{t-1}, A^t), \text{GCN}(X^t, A^t)]\right)
$$

$$
r_t = \sigma\left(W_r [\text{GCN}(h_{t-1}, A^t), \text{GCN}(X^t, A^t)]\right)
$$

$$
\tilde{h}_t = \tanh\left(W [r_t \odot \text{GCN}(h_{t-1}, A^t), \text{GCN}(X^t, A^t)]\right)
$$

$$
h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t
$$

El resultado $ h_t \in \mathbb{R}^{|V^t| \times d_{\text{hidden}}} $ representa el embedding dinámico de cada nodo en el instante $ t $, incorporando:

- Estructura relacional del grafo en $ t $ (vía GCN)
- Evolución temporal de dicha estructura (vía GRU)

En nuestro caso, $h_t$ es el embedding relacional de cada banco en t, el cual definimos como `e_rel`, el cual captura simultáneamente la posición del banco en la red de relaciones `MERG` en `t` (GCN) y la evolución temporal de esa posición a lo largo de los trimestres anteriores (GRU).

EvolveGCN (Pareja et al., 2020) propone una formulación distinta en la que no se modela la dinámica de los nodos directamente, sino la evolución de los pesos de la GCN:

$$
W_t^{(l)} = \text{GRU}(W_{t-1}^{(l)})
$$

$$
H_t^{(l+1)} = \sigma(\tilde{A}_t H_t^{(l)} W_t^{(l)})
$$

Este enfoque permite adaptarse de forma natural a la aparición y desaparición de nodos entre snapshots, ya que los parámetros del modelo evolucionan independientemente del conjunto de nodos activo. En el contexto bancario, donde el conjunto  $V^t$  varía por fusiones, adquisiciones y quiebras, este enfoque es particularmente relevante. Sin embargo, su coste computacional es mayor.

Para nuestro problema, T-GCN es una elección inicial más adecuada debido a que los cambios estructurales entre snapshots son moderados y existe evidencia empírica de su eficacia en problemas de early warning financiero.

## Definicion de atributos de nodo 
Para la composición del vector $X^t$ tenemos que para cada banco i en el trimestre t, el vector de atributos de nodo será la concatenación:
$$
x_i^t = [e_{i,\text{dev}}^{t} \; \Vert \; s_i]
$$
donde:

- $ e_{i,\text{dev}}^{t} \in \mathbb{R}^{192} $ es el embedding de desarrollo `TabPFN` del banco $ i $, almacenado en `embeddings_desarrollo.parquet`.
- $ s_i \in \mathbb{R}^k $ representa atributos estructurales y financieros, es decir, variables numéricas directas (OFFDOM, OFFTOT, OFFSTATE, coordenadas geográficas) y variables categóricas codificadas (BKCLASS, INSTTYPE, CHRTAGNT, REGAGNT, STALP) provenientes de `panel_nodos`.

Las variables categóricas deben ser codificadas numéricamente (por ejemplo, label encoding), ya que el GCN aprende representaciones latentes.

---

## Construcción de aristas desde MERG 

Como ya hemos visto por el estudio de las bases de datos, `MERG` registra eventos de fusión y adquisición. La construcción de $ E^t $ a partir de los eventos `MERG` es una decisión de diseño crítica. Tenemos dos enfoques principales a que podemos considerar:

- Enfoque A: Crear aristas de adquisición directa: $(i, j) \in E^t$ si y solo si $i$ adquirió $j$ en algún momento hasta $t$. Esto crea un grafo dirigido y disperso. El problema es que después de la adquisición el banco (nodo) $ j $ puede desaparecer de $ V^t $, lo que genera aristas huérfanas.

- Enfoque B — aristas de co-pertenencia a holding: dos bancos i,j comparten arista si tienen el mismo `RSSDHCR` (holding company), es decir $(i, j) \in E^t \iff \text{RSSDHCR}_i = \text{RSSDHCR}_j$. Esto es más preciso porque captura relaciones de riesgo sistémico real ya que si un holding tiene problemas, todos sus bancos están expuestos. `RSSDHCR` está en `panel_nodos` con ~50k `NaN` que corresponden a bancos independientes, es decir, sin holding, lo que tiene interpretación directa como nodos aislados.


En nuestro caso el enfoque B es mas acertado porque las relaciones de holding son el canal de contagio financiero más documentado en la literatura, y los `NaN` en `RSSDHCR` tienen interpretación directa (banco independiente = nodo aislado, sin aristas de este tipo). Tambien podemos considerar una combinación de ambas, es decir, aristas de holding con peso 1 y aristas de adquisición reciente con peso distinto $ w > 1 $, lo que induce una matriz de adyacencia ponderada:

$$
A^t \in \mathbb{R}^{|V^t| \times |V^t|}
$$

creando asi un grafo ponderado en lugar de binario.

## Función de pérdida con desbalanceo extremo
Dado el fuerte desbalance de clases (≈0.044% de positivos), la función de pérdida estándar (BCE) no es adecuada. La idea es tomar la función de perdida de `Focal Loss` (Lin et al., 2017), definida por_
$$
\text{FL}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)
$$
donde $\gamma > 0$ controla la reducción del peso en ejemplos fáciles (los negativos bien clasificados) y $\alpha_t$ balancea la contribución de clases (positivos/negativos). Alternativamente podemos tambien considerear la función de perdidad de BCE con ponderación de pesos, $\text{pos\_weight} = \frac{n_{\text{neg}}}{n_{\text{pos}}}$ ya que es más simple debido a que multiplica la pérdida de los positivos por $\text{pos\_weight} = \frac{n_{negativos}}{n_{positivos}} ≈ 2.270$. Será la función de partida antes de probar el `Focal Loss`.

Con todo esto visto, la integración en el pipeline final se define como:

1. Construcción de features:
   $$
   X^t = [e_{\text{TabPFN}}^t \Vert s^t]
   $$

2. Construcción del grafo:
   $$
   A^t \leftarrow \text{RSSDHCR} + \text{MERG}
   $$

3. Modelado temporal:
   T-GCN procesa la secuencia:
   $$
   \{(X^1, A^1), \dots, (X^T, A^T)\}
   $$
   produciendo embeddings dinámicos:
   $$
   e_{\text{rel}}^t \in \mathbb{R}^{|V^t| \times d_{\text{hidden}}}
   $$

4. Fusión de representaciones:
   $$
   e_{\text{hybrid}}^t = [e_{\text{Tab}}^t \Vert e_{\text{rel}}^t]
   $$

5. Modelado downstream:
   - MLP para scoring de riesgo
   - LSTM Autoencoder sobre trayectorias $\{e_{\text{hybrid}}^t\}$ para detección de deterioro temporal

Este esquema integra información estructural, temporal y latente en un único marco de aprendizaje jerárquico para análisis de riesgo bancario.

## Analisis del panel de nodos

El primer paso para la construción de nuestro modelo T-GCN sera explorar en más detalle nuestro panel de nodos.

In [1]:
import pandas as pd

df = pd.read_parquet('D:/financial_risk_data/processed/panel_nodos.parquet')
print('Shape:', df.shape)
print()
print('Columnas y dtypes:')
print(df.dtypes)
print()
print('NaN por columna:')
print(df.isnull().sum())
print()
print('Valores únicos BKCLASS:', df['BKCLASS'].unique())
print('Valores únicos INSTTYPE:', df['INSTTYPE'].unique() if 'INSTTYPE' in df.columns else 'no existe')
print('RSSDHCR no nulo:', df['RSSDHCR'].notna().sum(), '/', len(df))
print()
print(df.head(3).to_string())

Shape: (219576, 23)

Columnas y dtypes:
CERT              object
period            object
BKCLASS           object
INSTTYPE          object
CHRTAGNT          object
REGAGNT           object
STALP             object
SIMS_LAT         float64
SIMS_LONG        float64
CBSA             float64
METRO            float64
RSSDHCR          float64
ACTIVE             int64
FAILED             int64
DENOVO             int64
OFFDOM            object
OFFTOT            object
OFFSTATE         float64
closure_event    float64
closure_type     float64
fdic_assisted    float64
last_assets      float64
last_deposits    float64
dtype: object

NaN por columna:
CERT                 0
period               0
BKCLASS              0
INSTTYPE             0
CHRTAGNT             0
REGAGNT              0
STALP             1160
SIMS_LAT          1324
SIMS_LONG         1324
CBSA             57466
METRO            57466
RSSDHCR          49905
ACTIVE               0
FAILED               0
DENOVO               0
OFFDOM  

Tenemos 219.576 observaciones banco-trimestre con 23 columnas. Más que un simple panel_nodos es un panel espacio-temporal, donde cada fila es un banco i en un trimestre t, lo que significa que la dimensión temporal ya está incorporada. Esto es exactamente lo que necesitamos para construir los `snapshots` $G^t$.

Las columnas `closure_event`, `closure_type`, `fdic_assisted`, `last_assets`, `last_deposits` tienen 0 `NaN` pero valores 0.0 para bancos activos, son atributos del evento de cierre que solo toman valores distintos de cero en el trimestre de quiebra. Estas columnas no deben entrar como atributos de nodo al `T-GCN` porque introducirían data leakage directo haciendo que el modelo viera la quiebra antes de que ocurra.

Para la creación de las aristas (conjunto $E^t$) tomaremos como estructura inicial las relaciones descritas en `RSSDHCR`. Este tiene 169.671 valores no nulos sobre 219.576, es decir el 77.3% de las observaciones pertenecen a un holding. Los 49.905 NaN son bancos independientes, que seran nodos aislados sin aristas de holding.

__Observación:__ `RSSDHCR` es `float64` porque pandas convierte enteros con `NaN` a `float`. El valor real es un identificador entero del holding. 

Luego podemos definir la siguietne relación fundamental a la hora de la construcción de las aristas entre los nodos.
> Definición: Dos bancos i,j comparten arista en t si y solo si:
> $$
> RSSDHCR_i^t = RSSDHCR_j^t ≠ NaN
> $$

Esto define un grafo no dirigido donde las componentes conexas son exactamente los holdings. Dentro de cada holding el grafo es completo, todos los bancos del holding están conectados entre sí. Esto es correcto porque la exposición al riesgo del holding es simétrica entre sus miembros. La densidad del grafo depende del tamaño de los holdings. Si el holding medio tiene k bancos, cada nodo tiene grado k-1 en promedio. Los holdings grandes (k >> 1) generan cliques densas localmente pero el grafo global sigue siendo disperso porque hay miles de holdings distintos.

`RSSDHCR` es el `RSSDID` del _holding company_ regulatorio de nivel más alto, ya sea propietario directo o indirecto. Esto viene de la base de datos `NIC` (National Information Center) de la Reserva Federal, no de la FDIC directamente. El matiz crítico es "high holder". No es el holding inmediato del banco sino el propietario último en la cadena de control. Esto es exactamente lo que necesitamos para modelar riesgo sistémico porque dos bancos que comparten `RSSDHCR` comparten el mismo propietario último, aunque estén en holdings intermedios distintos. La exposición al riesgo de la cúspide de la cadena de control es la misma para ambos. Esto captura contagio a través de estructuras corporativas complejas que el holding inmediato no vería.

Desde el punto de vista de teoría de grafos, las componentes conexas de $E^t$ definidas por `RSSDHCR` son exactamente los conglomerados financieros regulatorios, que es la unidad de riesgo sistémico más relevante para nuestro problema.

Como atributos de nodos, en el conjunto $X^t$, de las 23 columnas solo entran como atributos de nodo $s_i^t$ aquellas que tienen información estructural sin introducir data leakage. Definimos una clasificación en tres grupos:

1. Aquellas caracteristicas que se introducen directamente como numéricas continuas.

    - `SIMS_LAT`, `SIMS_LONG`: coordenadas geográficas. Capturan exposición a riesgos regionales. Los 1.324 `NaN` se imputan con la media del estado (`STALP`).

    - `OFFDOM`, `OFFTOT`: número de oficinas. Están representadas como `object` en el panel de nodos, probablemente debido a que emplean comas como separadores de miles, lo cual hace que pandas las defina como `object`, luego necesitaran de una limpieza previa para convertirlos a `int`. Los 1.823 `NaN` se imputan con 0 o mediana.

    - `OFFSTATE` como float64 tiene 1.110 NaN, misma imputación que con el numero de oficinas.


2. Caracteristicas que se introducen codificadas como enteros (label encoding).
    - `BKCLASS`: 8 categorías: SM, NM, N, NC, SB, SI, SL, OI. Codifica el tipo de charter bancario. SM es state member bank, NM es national member, N es national bank, etc. Tiene poder predictivo porque distintas clases tienen distintos requisitos de capital y supervisión.
    - `INSTTYPE`: 3 categorías: CFR, TFR, FOR. Commercial Federal Reserve, Thrift Federal Reserve, Foreign. Relevante porque los thrifts (TFR) tuvieron tasas de quiebra distintas a los bancos comerciales
    - `CHRTAGNT` agente de charter: OCC = COMPTROLLER OF THE CURRENCY, OTS = OFFICE OF THRIFT SUPERVISION, STATE = STATE AGENCY y SOVER = FOREIGN COUNTRY. Determina el regulador primario.
    - `REGAGNT` — agente regulador: FED, FDIC, OCC, OTS. Correlaciona con `CHRTAGNT` pero no perfectamente.
    - `STALP` — estado federal (2 letras). 1.160 NaN que se imputan con la moda del `CERT` a lo largo del tiempo ya que el estado de un banco no cambia.

3. Caracteristicas que se introducen como binarias directamente:
    - `ACTIVE` — siempre 1 por construcción si el banco está en $V^t$, así que no aporta información y se descarta
    - `DENOVO` — banco de nova creación (primeros 3 años). Binaria, entra directamente
    - `METRO` — área metropolitana. Binaria con NaN coincidentes con CBSA (bancos rurales). NaN = 0


Las variables que no introducimos son las que nos producen un data leakge o son redundantes por su poca información que aportan, como:
- `FAILED` — es el target, no entra nunca como feature
- `CBSA` — código de área estadística metropolitana. Tiene 57.466 NaN y es redundante con METRO y coordenadas geográficas. Se descarta
- `RSSDHCR` — define las aristas, no entra como atributo de nodo para evitar que el modelo vea directamente la estructura que ya está codificada en Aᵗ
- `closure_event`, `closure_type`, `fdic_assisted` — leakage directo
- `last_assets`, `last_deposits` — leakage: solo tienen valor en el trimestre de quiebra

El vector de caracteristicas resultante que define los nodos de nuestro grafo posee una dimensión de $s_i^t \in \mathbb{R}^{12}$, distribuidas en 5 caracteristicas continuas, 5 caracteristicas label encoded y dos caracteristicas binarias. 

Finalmente el vector de atributos de nodo $x_i^t \in X^t$, es el resultado de la combinación del vector de caracteristicas definido anteriormente $s_i^t$, junto con la información adicional de `e_desarrollo` de `TabPFN` para un instante $t$. Es decir:
$$
x_i^t = [e_{i.dev}^t\; \Vert \; s_i^t] \in \mathbb{R}^{204}
$$
para todo $i\in \mathbb{N}$ (indice del banco) y $t \in \{1,2,\dots, T\}$ (trimestre temporal).

## Creación del archivo graph_builder.py

Con todo esto ya definido y expuesto, en nuestro archivo `graph_builder.py`, se define el constructor del grafo, el cual creara para cada trimestre $t$ un objeto `torch_geometric.data.Data` con:
```python
data.x          — tensor (|Vᵗ|, 204)   atributos de nodo
data.edge_index — tensor (2, |Eᵗ|)     aristas en formato COO
data.edge_attr  — tensor (|Eᵗ|, 1)     pesos (1.0 binario por ahora)
data.y          — tensor (|Vᵗ|,)       etiquetas failed (solo desarrollo)
data.cert       — lista de CERT        para alinear con embeddings
data.period     — string               trimestre
```
La secuencia completa de objetos `Data` forma el grafo dinámico que alimenta el T-GCN. Antes de construir nada tengamos en cuenta dos aspectos determinantes. El primero es que `RSSDHCR` se define en el panel de nodos como `float64` debido a los valores NaN en pandas. El valor real es un entero que identifica unívocamente al high holder en el sistema NIC de la Fed. Antes de construir aristas hay que convertirlo a `int64`. De esta forma la arista $(i,j) \in E^t$ se define como:
$$
RSSDHCR_{~i}^{~t} = RSSDHCR_{~j}^{~t} 
$$

donde ambos son dos valores no nulos `NaN`. De esta forma el grafo captura exactamente la estructura de conglomerados financieros regulatorios trimestre a trimestre. Ademas los embeddings en `embeddings_desarrollo.parquet `tienen 125.575 filas indexadas por `CERT` y `period`. El panel de nodos tiene 219.576 filas, luego la intersección por `(CERT, period)` define exactamente $V^t$ para cada t, es decir solo los bancos que tienen _embedding_ y _atributos estructurales_ en ese trimestre $t$. Por tanto antes de construir ningún `snapshot` $G^t$ hay que verificar que no hay bancos en `embeddings_desarrollo` que no estén en el panel de nodos y viceversa.


Antes de crear nada veamos la alineación entre _embeddings_ (tanto el de desarrollo como el de evaluación) y el panel de nodos.

In [3]:
import pandas as pd

emb_dev = pd.read_parquet('D:/financial_risk_data/embeddings/embeddings_desarrollo.parquet')
emb_eval = pd.read_parquet('D:/financial_risk_data/embeddings/embeddings_evaluacion.parquet')
nodos = pd.read_parquet('D:/financial_risk_data/processed/panel_nodos.parquet')

print('=== SHAPES ===')
print(f'emb_desarrollo : {emb_dev.shape}')
print(f'emb_evaluacion : {emb_eval.shape}')
print(f'panel_nodos    : {nodos.shape}')

print('\n=== PERIODOS ===')
periodos_dev   = set(emb_dev['period'].unique())
periodos_eval  = set(emb_eval['period'].unique())
periodos_nodos = set(nodos['period'].unique())

print('En dev no en nodos:', periodos_dev - periodos_nodos)
print('En eval no en nodos:', periodos_eval - periodos_nodos)
print('En nodos no en dev ni eval:', periodos_nodos - periodos_dev - periodos_eval)

print('\n=== ALINEACIÓN DESARROLLO ===')
dev_keys   = set(zip(emb_dev['CERT'].astype(str), emb_dev['period']))
nodos_keys = set(zip(nodos['CERT'].astype(str), nodos['period']))
print(f'Pares solo en emb_dev   : {len(dev_keys - nodos_keys)}')
print(f'Pares solo en nodos     : {len(nodos_keys - dev_keys)}')
print(f'Pares en común          : {len(dev_keys & nodos_keys)}')

print('\n=== ALINEACIÓN EVALUACIÓN ===')
eval_keys = set(zip(emb_eval['CERT'].astype(str), emb_eval['period']))
print(f'Pares solo en emb_eval  : {len(eval_keys - nodos_keys)}')
print(f'Pares solo en nodos     : {len(nodos_keys - eval_keys)}')
print(f'Pares en común          : {len(eval_keys & nodos_keys)}')

print('\n=== OFFDOM / OFFTOT (tipo object) ===')
print('Ejemplos OFFDOM:', nodos['OFFDOM'].dropna().head(5).tolist())
print('Ejemplos OFFTOT:', nodos['OFFTOT'].dropna().head(5).tolist())

=== SHAPES ===
emb_desarrollo : (125575, 194)
emb_evaluacion : (74361, 194)
panel_nodos    : (219576, 23)

=== PERIODOS ===
En dev no en nodos: set()
En eval no en nodos: set()
En nodos no en dev ni eval: {'2016Q1'}

=== ALINEACIÓN DESARROLLO ===
Pares solo en emb_dev   : 0
Pares solo en nodos     : 94001
Pares en común          : 125575

=== ALINEACIÓN EVALUACIÓN ===
Pares solo en emb_eval  : 0
Pares solo en nodos     : 145215
Pares en común          : 74361

=== OFFDOM / OFFTOT (tipo object) ===
Ejemplos OFFDOM: ['3', '9', '6', '4', '5']
Ejemplos OFFTOT: ['4', '9', '6', '4', '5']


El resultado es el esperado, `2016Q1` está en `panel_nodos` pero no en ningún `embedding` ya que es el trimestre omitido en el loop de desarrollo porque no tiene contexto previo. En la fase de desarrollo (train) 0 pares solo en `e_dev` significa que todos los bancos con `embedding` tienen sus atributos estructurales en `panel_nodos`. Los 94.001 pares solo en nodos son los trimestres que `panel_nodos` tiene pero que no están en el embedding de desarrollo ya que este panel incluye 2016Q1 más todos los trimestres del bloque de evaluación.

En la fase de evaluación tenemos el mismo patrón, 0 pares solo en e_eval. Los 145.215 pares solo en nodos son el bloque de desarrollo más 2016Q1. `OFFDOM` y `OFFTOT` son `strings` de enteros sin comas, luego la conversión es directa con `astype(int)`, no hay separadores de miles que limpiar.

Estos resultados nos confirman que el `join` definido es correcto por construcción. $V^t$ para cada trimestre $t$ es exactamente el conjunto de bancos en `e_desarrollo` (`e_dev`) o `e_evaluacion` para dicho $t$. No hay bancos con _embedding_ sin atributos estructurales, por tanto el `join` se hace siempre por `(CERT, period)` sin pérdida estructural.